# 03 — Retrieval Strategies

Four approaches to finding relevant context in the vector store:
1. **Semantic search** — find by meaning, optionally with manual metadata filters
2. **SelfQueryRetriever** — LLM automatically extracts filters from the question
3. **Temporal comparison** — per-quarter retrieval for cross-time analysis
4. **Hybrid search** — combines keyword matching (BM25) with semantic search for the best of both worlds

In [ ]:
import sys
sys.path.insert(0, "..")

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("../.env")

In [ ]:
# ChromaDB (raw client for direct queries)
chroma_embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(
    name="earnings_calls", embedding_function=chroma_embedding_fn,
)

# LangChain vectorstore wrapper (needed for SelfQueryRetriever)
lc_embedding_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(
    collection_name="earnings_calls",
    embedding_function=lc_embedding_fn,
    persist_directory="../chroma_db",
)

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Available quarters
all_meta = collection.get(include=["metadatas"])
quarters = sorted(set(m["quarter"] for m in all_meta["metadatas"]))
print(f"Collection: {collection.count()} documents | Quarters: {quarters}")

In [ ]:
def show_results(results, max_text=150):
    """Display ChromaDB query results."""
    for i in range(len(results["documents"][0])):
        meta = results["metadatas"][0][i]
        dist = results["distances"][0][i]
        print(f"[{meta['quarter']}] {meta['speaker']} ({meta.get('role', '')}) — dist: {dist:.3f}")
        print(f"  {results['documents'][0][i][:max_text]}...\n")

def show_docs(docs, max_text=150):
    """Display LangChain Document results."""
    for i, doc in enumerate(docs):
        meta = doc.metadata
        print(f"[{meta.get('quarter')}] {meta.get('speaker')} ({meta.get('role', '')})")
        print(f"  {doc.page_content[:max_text]}...\n")

---
## 1. Semantic Search + Manual Filters

The simplest approach: query by meaning, optionally narrow results with `where` filters. You write the filters yourself, which gives full control but requires knowing the exact field names and values. Good for precise, targeted queries.

In [ ]:
# Pure semantic search — no filters
print("Query: 'What were the revenue results?' (no filter)\n")
show_results(collection.query(query_texts=["What were the revenue results?"], n_results=3))

In [ ]:
# Filtered by quarter
print("Query: 'What were the revenue results?' (Q4-2025 only)\n")
show_results(collection.query(
    query_texts=["What were the revenue results?"],
    n_results=3,
    where={"quarter": "Q4-2025"},
))

In [ ]:
# Compound filter: CEO + specific quarter
print("Query: 'emerging markets' (CEO + Q3-2025)\n")
show_results(collection.query(
    query_texts=["emerging markets growth"],
    n_results=3,
    where={"$and": [{"role": "Chief Executive Officer"}, {"quarter": "Q3-2025"}]},
))

In [ ]:
# Off-topic query — retriever still returns results but distances are high
print("Query: 'electric vehicle strategy' (off-topic)\n")
results = collection.query(query_texts=["electric vehicle strategy"], n_results=3)
show_results(results)
print(f"Note: distances are {[f'{d:.2f}' for d in results['distances'][0]]} — high means low relevance.")

---
## 2. SelfQueryRetriever

Instead of writing filters manually, the LLM parses the natural language question and extracts filters automatically. It needs a schema (`AttributeInfo`) describing what fields exist and what values they can take — this is what the LLM reads to decide when to filter.

*"What did Apple's CFO say about margins in Q3 2025?"* becomes:
- Semantic query: `"margins"`
- Filters: `company=AAPL`, `quarter=Q3-2025`, `role=Chief Financial Officer`

In [ ]:
metadata_field_info = [
    AttributeInfo(name="company", type="string",
        description="Stock ticker symbol. Values: AAPL (Apple)"),
    AttributeInfo(name="quarter", type="string",
        description=f"Fiscal quarter in Q#-YYYY format. Values: {', '.join(quarters)}"),
    AttributeInfo(name="speaker", type="string",
        description="Full name of the speaker. Examples: Timothy D. Cook, Kevan Parekh"),
    AttributeInfo(name="role", type="string",
        description="Job title. Examples: Chief Executive Officer, Chief Financial Officer. Empty for analysts."),
]

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Transcripts of quarterly earnings call presentations and Q&A sessions",
    metadata_field_info=metadata_field_info,
)
print("SelfQueryRetriever ready")

In [ ]:
# The LLM should extract: company=AAPL, quarter=Q4-2025
print("Query: 'What were Apple's revenue results in Q4 2025?'\n")
show_docs(retriever.invoke("What were Apple's revenue results in Q4 2025?"))

In [ ]:
# The LLM should extract: role=CFO
print("Query: 'What did the CFO say about gross margins?'\n")
show_docs(retriever.invoke("What did the CFO say about gross margins?"))

In [ ]:
# No filters needed — pure semantic
print("Query: 'What is the company strategy for AI?'\n")
show_docs(retriever.invoke("What is the company strategy for AI?"))

---
## 3. Temporal Comparison

A question like *"How has revenue changed across quarters?"* can't be answered by a single retrieval — results tend to cluster in whichever quarter has the strongest semantic match, missing others entirely. The solution: retrieve **separately per quarter** to guarantee coverage, then let the LLM synthesize the comparison.

In [ ]:
# Problem: standard retrieval may cluster results in one quarter
results = collection.query(
    query_texts=["How has revenue changed across quarters?"], n_results=4,
)
result_quarters = [results["metadatas"][0][i]["quarter"] for i in range(len(results["documents"][0]))]
print(f"Standard retrieval quarters: {set(result_quarters)}")
print(f"Missing: {set(quarters) - set(result_quarters)}")

In [ ]:
def retrieve_per_quarter(query, quarters, n_per_quarter=2, company=None):
    """Retrieve top chunks for each quarter separately."""
    results_by_quarter = {}
    for quarter in quarters:
        where = {"quarter": quarter}
        if company:
            where = {"$and": [{"quarter": quarter}, {"company": company}]}
        results = collection.query(query_texts=[query], n_results=n_per_quarter, where=where)
        results_by_quarter[quarter] = [
            {
                "text": results["documents"][0][i],
                "speaker": results["metadatas"][0][i].get("speaker", "?"),
                "role": results["metadatas"][0][i].get("role", ""),
            }
            for i in range(len(results["documents"][0]))
        ]
    return results_by_quarter


def build_temporal_context(results_by_quarter):
    """Format per-quarter results with === Q#-YYYY === headers."""
    sections = []
    for quarter in sorted(results_by_quarter):
        chunks = []
        for d in results_by_quarter[quarter]:
            speaker = d["speaker"] + (f" ({d['role']})" if d["role"] else "")
            chunks.append(f"[{speaker}]: {d['text']}")
        sections.append(f"=== {quarter} ===\n" + "\n\n".join(chunks))
    return "\n\n".join(sections)

In [ ]:
# Per-quarter retrieval — every quarter is represented
results = retrieve_per_quarter("revenue results", quarters, company="AAPL")
for q, docs in results.items():
    print(f"{q}: {len(docs)} chunks")
    for d in docs:
        print(f"  [{d['speaker']}] {d['text'][:80]}...")
    print()

In [ ]:
# Temporal RAG: retrieve per quarter + LLM comparison
TEMPORAL_PROMPT = ChatPromptTemplate.from_template("""
You are an analyst assistant. The context below contains earnings call excerpts 
organized by quarter. Analyze the data across all quarters to identify trends 
and changes. Be specific with numbers. Note if information is missing for any quarter.

Context:
{context}

Question: {question}
""")

temporal_chain = TEMPORAL_PROMPT | llm | StrOutputParser()

def ask_temporal(question, quarters=None, company=None):
    if quarters is None:
        meta = collection.get(include=["metadatas"])
        quarters = sorted(set(m["quarter"] for m in meta["metadatas"]))
    results = retrieve_per_quarter(question, quarters, company=company)
    context = build_temporal_context(results)
    return temporal_chain.invoke({"context": context, "question": question})

print("Temporal RAG ready")

In [ ]:
q = "How has Apple's revenue changed across quarters?"
print(f"Q: {q}\n")
print(ask_temporal(q, company="AAPL"))

In [ ]:
q = "How has the discussion about AI evolved across quarters?"
print(f"Q: {q}\n")
print(ask_temporal(q, company="AAPL"))

---
## 4. Hybrid Search (BM25 + Semantic)

Semantic search finds by meaning but can miss exact matches — if someone asks about "46.9% gross margin", the embedding might not capture the exact number well. BM25 (keyword search) finds exact term matches but misses paraphrases. **Hybrid search combines both**: BM25 retrieves by keywords, semantic retrieves by meaning, and `EnsembleRetriever` merges and re-ranks the results.

In [ ]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Load all turns as LangChain Documents (BM25 needs the full corpus in memory)
processed_dir = Path("../data/processed")
all_documents = []
for path in sorted(processed_dir.glob("*.json")):
    with open(path, "r", encoding="utf-8") as f:
        transcript = json.load(f)
    for turn in transcript["turns"]:
        all_documents.append(Document(
            page_content=turn["text"],
            metadata={
                "company": transcript["company"],
                "quarter": transcript["quarter"],
                "speaker": turn["speaker"],
                "role": turn["role"],
            },
        ))

print(f"Loaded {len(all_documents)} documents for BM25")

In [ ]:
# BM25 retriever (keyword-based)
bm25_retriever = BM25Retriever.from_documents(all_documents, k=4)

# Semantic retriever (from the existing LangChain vectorstore)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Hybrid: combine both with equal weights
# Weights control the balance: 0.5/0.5 = equal, 0.7/0.3 = favor semantic
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.5, 0.5],
)

print("Hybrid retriever ready (BM25 0.5 + Semantic 0.5)")

### Compare: Semantic vs. BM25 vs. Hybrid

A query with specific numbers shows where semantic search struggles and BM25 shines — and vice versa.

In [ ]:
# Query with a specific number — BM25 should find this better
query = "What was the 46.5% gross margin?"

print("SEMANTIC only:")
for doc in semantic_retriever.invoke(query)[:3]:
    m = doc.metadata
    print(f"  [{m['quarter']}] {m['speaker']}: {doc.page_content[:100]}...")

print("\nBM25 only:")
for doc in bm25_retriever.invoke(query)[:3]:
    m = doc.metadata
    print(f"  [{m['quarter']}] {m['speaker']}: {doc.page_content[:100]}...")

print("\nHYBRID:")
for doc in hybrid_retriever.invoke(query)[:3]:
    m = doc.metadata
    print(f"  [{m['quarter']}] {m['speaker']}: {doc.page_content[:100]}...")

In [ ]:
# Query with paraphrase — semantic should find this better
query = "How is the company doing with its subscription services?"

print("SEMANTIC only:")
for doc in semantic_retriever.invoke(query)[:3]:
    m = doc.metadata
    print(f"  [{m['quarter']}] {m['speaker']}: {doc.page_content[:100]}...")

print("\nBM25 only:")
for doc in bm25_retriever.invoke(query)[:3]:
    m = doc.metadata
    print(f"  [{m['quarter']}] {m['speaker']}: {doc.page_content[:100]}...")

print("\nHYBRID:")
for doc in hybrid_retriever.invoke(query)[:3]:
    m = doc.metadata
    print(f"  [{m['quarter']}] {m['speaker']}: {doc.page_content[:100]}...")

In [ ]:
# Hybrid retriever works with the RAG chain too
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an analyst assistant answering questions about earnings calls.
Use ONLY the provided context. If the answer is not in the context, say so.
Cite the speaker and quarter when relevant.

Context:
{context}

Question: {question}
""")

hybrid_rag_chain = RAG_PROMPT | llm | StrOutputParser()

def ask_hybrid(question):
    docs = hybrid_retriever.invoke(question)
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('company')} {d.metadata.get('quarter')} — {d.metadata.get('speaker', '?')}]\n{d.page_content}"
        for d in docs
    )
    return hybrid_rag_chain.invoke({"context": context, "question": question})

q = "What specific gross margin percentage was reported and what drove it?"
print(f"Q: {q}\n")
print(ask_hybrid(q))